# ATLAS — Placement Outlook Baseline (Synthetic Data)

This notebook audits the supplied bundle, aggregates pre-outcome sources into one feature table, trains a Logistic Regression placement-outlook model, and compares it with a simple majority-class baseline.

**Important:** this bundle is synthetic. Metrics measure whether the pipeline can recover the simulated relationship; they are not real-world placement performance estimates. Demographics and Branch are intentionally excluded from model inputs. `Package_LPA`, offers and all other outcome-side columns must never be predictors.


## 1. Upload and extract the dataset ZIP
Run the cell and select `ATLAS_Realistic_Campus_Dataset.zip`. If you run this notebook outside Colab, set `DATASET_ZIP` to the path of the ZIP before running the cell.

In [ ]:
from pathlib import Path
import zipfile, os, sys, json

try:
    from google.colab import files
    uploaded = files.upload()
    zip_candidates = [name for name in uploaded if name.lower().endswith('.zip')]
    if not zip_candidates:
        raise ValueError('Upload the ATLAS dataset .zip file.')
    dataset_zip = zip_candidates[0]
except ImportError:
    dataset_zip = 'ATLAS_Realistic_Campus_Dataset.zip'  # edit when running locally

EXTRACT_ROOT = Path('/content/atlas_bundle') if Path('/content').exists() else Path('./atlas_bundle')
EXTRACT_ROOT.mkdir(parents=True, exist_ok=True)
with zipfile.ZipFile(dataset_zip, 'r') as zf:
    zf.extractall(EXTRACT_ROOT)

data_matches = list(EXTRACT_ROOT.rglob('student_master.csv'))
if not data_matches:
    raise FileNotFoundError('Could not find data/student_master.csv after extracting the ZIP.')
DATA_DIR = data_matches[0].parent
PROJECT_ROOT = DATA_DIR.parent
SCRIPTS_DIR = PROJECT_ROOT / 'scripts'
sys.path.insert(0, str(SCRIPTS_DIR))
print('Data directory:', DATA_DIR)
print('Project root:', PROJECT_ROOT)


## 2. Audit the source tables
Missing records and fields are intentional. The notebook does not convert absent LMS weeks or absent feedback into zero or dissatisfaction.

In [ ]:
import pandas as pd
from IPython.display import display

manifest_path = DATA_DIR / 'source_manifest.csv'
if manifest_path.exists():
    display(pd.read_csv(manifest_path))

master = pd.read_csv(DATA_DIR / 'student_master.csv')
outcomes = pd.read_csv(DATA_DIR / 'placement_outcomes.csv')
print(f"Students: {len(master):,}")
print(f"Unique student IDs: {master['Student_ID'].nunique():,}")
print(f"Duplicate master IDs: {master['Student_ID'].duplicated().sum():,}")
print('Placement outcome distribution:')
display(outcomes['Placement_Status'].value_counts(dropna=False).rename_axis('Placement_Status').to_frame('students'))
print(f"Simulated placed rate: {(outcomes['Placement_Status'] == 'Placed').mean():.1%}")


## 3. Build student-level features and train the model
The aggregation and safeguards live in the bundled `scripts/atlas_model_pipeline.py`. Preprocessing is fitted on the training fold only. The evaluation uses a held-out stratified split; the dummy classifier provides a simple comparison baseline.

In [ ]:
from atlas_model_pipeline import build_feature_table, train_baseline

X, y, outcome = build_feature_table(DATA_DIR)
print('Feature table shape:', X.shape)
print('Target rows:', len(y))
print('Demographic / branch columns in features:', [c for c in X.columns if c in {'Age','Gender','Home_State','Branch','College_Tier'}])
print('Outcome/package leakage columns in features:', [c for c in X.columns if any(x in c.lower() for x in ['package_lpa','placement_status','offer_count','first_offer_date','days_to_first_offer'])])

OUTPUT_DIR = PROJECT_ROOT / 'outputs'
model, metrics, coef_df, split = train_baseline(DATA_DIR, OUTPUT_DIR)
summary = pd.DataFrame([
    {'Model':'Logistic Regression','ROC-AUC':metrics['logistic_regression']['roc_auc'],
     'PR-AUC':metrics['logistic_regression']['pr_auc'],'Accuracy (threshold 0.5)':metrics['logistic_regression']['accuracy_at_0_5'],
     'Precision (threshold 0.5)':metrics['logistic_regression']['precision_at_0_5'],
     'Recall (threshold 0.5)':metrics['logistic_regression']['recall_at_0_5']},
    {'Model':'Dummy baseline','ROC-AUC':metrics['dummy_baseline']['roc_auc'],
     'PR-AUC':metrics['dummy_baseline']['pr_auc'],'Accuracy (threshold 0.5)':metrics['dummy_baseline']['accuracy'],
     'Precision (threshold 0.5)':None,'Recall (threshold 0.5)':None},
])
display(summary.round(3))
print('Saved outputs to:', OUTPUT_DIR)


## 4. Inspect strongest model associations
Coefficients are standardised model associations, not causal explanations. Use them as a diagnostic, not as proof that changing a feature will cause placement.

In [ ]:
display(coef_df.head(15).round(3))
print('Confusion matrix [actual rows: Not Placed, Placed; predicted columns: Not Placed, Placed]')
print(metrics['logistic_regression']['confusion_matrix'])


## 5. Save evaluation outputs
The pipeline, held-out predictions, metrics and coefficients are saved under `outputs/`. They belong to this synthetic experiment. Do not present the metrics as validated college performance.

In [ ]:
print(json.dumps({
    'provenance': metrics['dataset_provenance'],
    'students': metrics['rows_total'],
    'features': metrics['features_used'],
    'train_rows': metrics['train_rows'],
    'test_rows': metrics['test_rows'],
    'placement_rate': round(metrics['placement_rate_overall'], 4),
    'logistic_roc_auc': round(metrics['logistic_regression']['roc_auc'], 4),
    'dummy_roc_auc': round(metrics['dummy_baseline']['roc_auc'], 4),
}, indent=2))


## Before integrating into ATLAS
- Keep `Placement_Status` and outcome-side fields separate from inputs.
- Preserve the same feature engineering and preprocessing when scoring new students.
- Return a model version and feature timestamp with every score.
- Display probability estimates only as experimental until validated on authorised institutional outcomes.
- Do not use this synthetic model to make consequential student decisions.
